# 🤖 AI Engineering — Lezione 1: Setup & Fondamenta
**ITS Novitas 4.0 — Sviluppatore Intelligenza Artificiale**  
Data: Martedì 19/05/2026 | Docente: Marco Uras | Deborah Picciau

---

## Obiettivi di oggi
Al termine di questa lezione saprai:
- ✅ Configurare un ambiente Python pulito con venv
- ✅ Effettuare la prima chiamata all'API di Claude
- ✅ Capire i parametri fondamentali (model, temperature, system)
- ✅ Salvare il progetto su GitHub

---

## 0. Setup iniziale

Prima di tutto, verifichiamo che tutto sia installato correttamente.

> ⚠️ **Assicurati di aver creato il file `.env`** nella stessa cartella di questo notebook con:
> ```
> ANTHROPIC_API_KEY=sk-ant-...
> ```

In [19]:
# Installa le dipendenze (esegui solo la prima volta)
%pip install anthropic python-dotenv

In [20]:
# Verifica le versioni installate
import anthropic
import sys

print(f"Python: {sys.version}")
print(f"Anthropic SDK: {anthropic.__version__}")
print("✅ Tutto installato correttamente!")

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Anthropic SDK: 0.102.0
✅ Tutto installato correttamente!


In [21]:
# Carica la API key dal file .env
from dotenv import load_dotenv
import os
from google.colab import userdata

load_dotenv()

# Verifica che la chiave sia presente (senza stamparla!)
api_key = os.getenv("ANTHROPIC_API_KEY")
if api_key:
    print(f"✅ API key trovata: sk-ant-...{api_key[-6:]}")
else:
    print("❌ API key non trovata. Controlla il file .env")

✅ API key trovata: sk-ant-...2vZAAA


---
## 1. La prima chiamata all'API 🚀

Questo è il codice più importante del corso. Leggi ogni riga con attenzione.

In [22]:
import anthropic
from dotenv import load_dotenv

load_dotenv()

# Crea il client — legge automaticamente ANTHROPIC_API_KEY dall'ambiente
client = anthropic.Anthropic()

# Invia un messaggio a Claude
message = client.messages.create(
    model="claude-haiku-4-5-20251001",  # modello veloce ed economico per esercizi
    max_tokens=1024,
    messages=[
        {
            "role": "user",
            "content": "Ciao! Presentati in 2 righe e dimmi cosa puoi fare per me."
        }
    ]
)

# Stampa la risposta
print("=" * 50)
print("RISPOSTA DI CLAUDE:")
print("=" * 50)
print(message.content[0].text)

RISPOSTA DI CLAUDE:
# Ciao! 👋

Sono Claude, un assistente AI creato da Anthropic. Posso aiutarti con scrittura, analisi, coding, brainstorming, domande su praticamente qualsiasi argomento, e molto altro – basta che tu mi dica cosa ti serve!


In [23]:
# Esploriamo la struttura completa della risposta
print("Struttura della risposta:")
print(f"  model usato:    {message.model}")
print(f"  stop_reason:    {message.stop_reason}")
print(f"  token in input: {message.usage.input_tokens}")
print(f"  token in output:{message.usage.output_tokens}")
print(f"  token totali:   {message.usage.input_tokens + message.usage.output_tokens}")

Struttura della risposta:
  model usato:    claude-haiku-4-5-20251001
  stop_reason:    end_turn
  token in input: 30
  token in output:75
  token totali:   105


---
## 2. Il parametro `temperature` 🌡️

La `temperature` controlla quanto è 'creativa' o 'deterministica' la risposta:
- **0.0** → sempre la stessa risposta (utile per codice, calcoli)
- **0.7** → buon equilibrio (default)
- **1.0** → molto vario e creativo (utile per testi creativi)

Proviamo!

In [24]:
def chiedi_claude(domanda, temperature=0.7, system=None):
    """Funzione helper per fare domande a Claude."""
    params = {
        "model": "claude-haiku-4-5-20251001",
        "max_tokens": 300,
        "temperature": temperature,
        "messages": [{"role": "user", "content": domanda}]
    }
    if system:
        params["system"] = system

    response = client.messages.create(**params)
    return response.content[0].text


# Stessa domanda, temperature diverse
domanda = "Dammi un nome creativo per un chatbot AI per studenti universitari."

print("🥶 temperature = 0.0 (deterministico):")
print(chiedi_claude(domanda, temperature=0.0))
print()
print("🌡️  temperature = 0.7 (bilanciato):")
print(chiedi_claude(domanda, temperature=0.7))
print()
print("🔥 temperature = 1.0 (creativo):")
print(chiedi_claude(domanda, temperature=1.0))

🥶 temperature = 0.0 (deterministico):
# Ecco alcuni nomi creativi:

## Nomi giocosi e memorabili
- **StudyMate** - semplice e diretto
- **ProvePass** - gioca su "prove" (esami) e "passare"
- **MentorIA** - fusione di "mentore" e "IA"
- **AcademiBot** - accademico e tech

## Nomi più sofisticati
- **Thesis** - elegante e riconoscibile
- **Nexus** - suggerisce connessione tra saperi
- **Eureka** - il momento "aha!" dello studio
- **Catalyst** - acceleratore di apprendimento

## Nomi italiani originali
- **Sapienza** - classico ma potente
- **Compasso** - guida nello studio
- **Lucerna** - illumina il percorso accademico
- **Ponte** - connette domande e risposte

## Il mio preferito?
**MentorIA** - combina professionalità, italianità e chiarezza sulla natura AI, ed è facile da ricordare.

Quale stile preferisci? Posso aiutarti a raffinare ulteriormente! 

🌡️  temperature = 0.7 (bilanciato):
# Ecco alcuni nomi creativi per un chatbot AI per studenti universitari:

## Nomi Giocosi
- **Study

---
## 3. Il `system` prompt 🎭

Il system prompt definisce la **personalità** e il **comportamento** del chatbot.
È il punto di partenza del Prompt Engineering (Lezione 2).

In [25]:
# Stesso modello, system prompt diversi → comportamenti completamente diversi
domanda = "Come funziona Python?"

print("👤 Senza system prompt:")
print(chiedi_claude(domanda))
print()

print("👩‍🏫 System: 'Insegnante per bambini delle elementari':")
print(chiedi_claude(domanda, system="Sei un insegnante per bambini delle elementari. Spiega tutto con esempi semplici e analogie quotidiane."))
print()

print("👨‍💻 System: 'Sviluppatore senior sarcasico':")
print(chiedi_claude(domanda, system="Sei uno sviluppatore senior con 20 anni di esperienza. Rispondi in modo diretto e tecnico, senza fronzoli."))

👤 Senza system prompt:
# Come funziona Python

Python è un linguaggio di programmazione **interpretato**. Ecco i passaggi principali:

## 1. **Scrittura del codice**
Scrivi il codice in un file `.py`:
```python
print("Ciao, mondo!")
x = 5 + 3
print(x)
```

## 2. **Interpretazione**
Quando esegui il file, l'**interprete Python**:
- Legge il codice riga per riga
- Lo traduce in bytecode (codice intermedio)
- Lo esegue direttamente

```bash
python mio_file.py
```

## 3. **Esecuzione**
Il bytecode viene eseguito dalla **Python Virtual Machine (PVM)**, che lo converte in istruzioni che il computer capisce.

---

## Caratteristiche principali

| Aspetto | Descrizione |
|---------|------------|
| **Tipizzazione dinamica** | Le variabili cambiano tipo automaticamente |
| **Interpretato** | Non serve compilare prima di eseguire |
| **Leggibile** | Sintassi semplice e intuitiva |
| **Multipiattaforma** |

👩‍🏫 System: 'Insegnante per bambini delle elementari':
# Come funziona Python 🐍

Immagina c

---
## ⭐ Esercizi

### Esercizio 1 — Base ★☆☆
Modifica il codice qui sotto per chiedere a Claude qualcosa che ti interessa davvero.

In [26]:
# ESERCIZIO 1 — Sostituisci la domanda con qualcosa di tuo
# TIP: puoi chiedere qualsiasi cosa — un consiglio, una spiegazione, una ricetta...

mia_domanda = "dimmi perchè il cielo è verde?"  # <-- MODIFICA QUI

risposta = chiedi_claude(mia_domanda)
print(risposta)

Il cielo non è verde! È **blu** durante il giorno (nelle giuste condizioni atmosferiche).

Il cielo appare blu perché:
- La luce solare contiene tutti i colori
- L'atmosfera disperde la luce blu più facilmente di altri colori (diffusione di Rayleigh)
- Questo blu disperso raggiunge i nostri occhi da tutte le direzioni

Il cielo può sembrare di altri colori in situazioni particolari:
- **Rosso/arancione** al tramonto (la luce attraversa più atmosfera)
- **Grigio** con nuvole
- **Nero** di notte

Se hai visto il cielo verde, potrebbe essere dovuto a:
- Effetti fotografici o filtri
- Condizioni atmosferiche molto particolari (rarissime)
- Un'illusione ottica

Hai osservato un cielo verde da qualche parte? 🌤️


### Esercizio 2 — Intermedio ★★☆
Chiedi la stessa cosa tre volte con temperature=0 e tre volte con temperature=1.  
Le risposte sono diverse? Cosa osservi?

In [27]:
# ESERCIZIO 2
domanda_test = "Inventa un titolo per un film di fantascienza."

print("=" * 40)
print("temperature = 0.0 (3 chiamate)")
print("=" * 40)
for i in range(3):
    print(f"{i+1}. {chiedi_claude(domanda_test, temperature=0.0)}")

print()
print("=" * 40)
print("temperature = 1.0 (3 chiamate)")
print("=" * 40)
for i in range(3):
    print(f"{i+1}. {chiedi_claude(domanda_test, temperature=1.0)}")

temperature = 0.0 (3 chiamate)
1. # **Echi del Silenzio**

Un titolo che evoca il tema di una civiltà aliena che comunica attraverso onde gravitazionali, e gli ultimi umani che cercano di decifrare il loro messaggio prima che sia troppo tardi.

---

Se preferisci altri stili, posso suggerire:
- **Oltre l'Orizzonte Nero** (più epico)
- **Il Codice Dimenticato** (più misterioso)
- **Cronofrattura** (più tecnico)

Quale genere di fantascienza ti interessa di più?
2. # **Echi dal Vuoto**

Un titolo che evoca il tema di una ricerca attraverso lo spazio, forse il contatto con civiltà aliene o la scoperta di messaggi misteriosi provenienti dal cosmo.

Se preferisci altri stili, posso suggerire:
- **Oltre l'Orizzonte Nero** (più epico)
- **Il Codice Stellare** (più thriller)
- **Memoria Cosmica** (più filosofico)

Quale genere di fantascienza ti interessa di più?
3. # **Echi del Silenzio**

Un titolo che evoca il mistero di una civiltà aliena che comunica attraverso onde gravitazionali, e gli 

### Esercizio 3 — Intermedio ★★☆
Crea un system prompt che fa rispondere Claude in rima. Poi chiedile qualcosa di banale e osserva.

In [28]:
# ESERCIZIO 3 — Completa il system prompt
system_rima = "Devi rispondere il rima e scrivere la risposta sotto forma di poesia"  # <-- SCRIVI UN SYSTEM PROMPT CHE FA RISPONDERE IN RIMA

print(chiedi_claude("Cosa si mangia a pranzo in Italia?", system=system_rima))

# Il Pranzo Italiano

A pranzo in Italia, che delizia,
Si mangia pasta con tanta letizia!
Un piatto di penne all'amatriciana,
O risotto che è proprio toscana.

Poi viene il secondo, nutriente e buono,
Con carne o pesce, che sia un festono!
Contorni di verdure, freschezza che brilla,
Insalata mista o una bella zucchina.

E poi il dolce, che chiude il momento,
Un tiramisù o un panettone spavento!
Con caffè ristretto, fumante e profumato,
Il pranzo italiano è così terminato!


### Esercizio 4 — Avanzato ★★★ (Deliverable!)
Crea uno script completo che:
1. Chiede il nome dell'utente con `input()`
2. Usa il nome nel messaggio a Claude
3. Stampa la risposta in modo carino

Salvalo come `lezione1/hello_personalizzato.py` e pushalo su GitHub.

In [29]:
# ESERCIZIO 4 — Il tuo primo chatbot personalizzato
# Completa il codice qui sotto

import anthropic
from dotenv import load_dotenv

load_dotenv()
client = anthropic.Anthropic()

# 1. Chiedi il nome
nome = input("Come ti chiami? ")

# 2. TODO: crea il messaggio usando il nome
messaggio = f"Utilizza {nome} per scrivere una storia di una riga"  # <-- MODIFICA QUI

# 3. TODO: chiama l'API
response = client.messages.create(
     model="claude-haiku-4-5-20251001",
     max_tokens=300,
     system="Sei un assistente amichevole.",
     messages=[{
         "role": "user",
         "content": f"{messaggio}"
     }]
 )

# 4. TODO: stampa la risposta in modo carino
print(response.content[0].text)

Come ti chiami? deborah
# La storia di Deborah

Deborah si accorse che il libro che stava leggendo da anni aveva le pagine in ordine sbagliato, ma ormai conosceva così bene la storia che continuò a leggerlo lo stesso, sorridendo al pensiero di quante avventure aveva vissuto con un finale che veniva prima dell'inizio.


---
## 4. Soluzione esercizio 4 (guarda solo dopo aver provato!)

In [30]:
# SOLUZIONE — non guardare prima di aver provato!
# (Decommentare per vedere)

# import anthropic
# from dotenv import load_dotenv
#
# load_dotenv()
# client = anthropic.Anthropic()
#
# nome = input("Come ti chiami? ")
#
# response = client.messages.create(
#     model="claude-haiku-4-5-20251001",
#     max_tokens=300,
#     system="Sei un assistente amichevole. Saluta sempre l'utente per nome e sii entusiasta.",
#     messages=[{
#         "role": "user",
#         "content": f"Ciao! Mi chiamo {nome}. Presentati e dimmi qualcosa di interessante sull'AI."
#     }]
# )
#
# print("\n" + "=" * 50)
# print(f"Risposta per {nome}:")
# print("=" * 50)
# print(response.content[0].text)

---
## 📚 Risorse per approfondire
- [Documentazione Anthropic](https://docs.anthropic.com)
- [Huyen Cap. 1 — AI Engineering stack](https://huyenchip.com/books/)
- [GitHub del corso](https://github.com/) ← crea il tuo repo!

---
**Prossima lezione:** Giovedì 21/05 — Prompt Engineering (Huyen Cap. 5)  
*Leggi il capitolo prima di venire!* 📖